# Q2 - Multiclass boosting from scratch (SAMME) on Olivetti faces
Run top to bottom. This notebook re-uses the saved split (`split/split_indices.npz`) and the Q1 frozen tree setting / final B (`logs/`). If those Q1 files are missing it recomputes them with the same rules as Q1.

In [ ]:
# ---- CONFIG ----
ROLL_NO = "0202"
SEED = int("".join(ch for ch in ROLL_NO if ch.isdigit()))     # 202
K = 40                                  # number of classes
DEPTHS = [1, 2, 3, 4]                   # boosting weak-learner depths
ROUNDS = [10, 25, 50, 100]
T_MAX = max(ROUNDS)
CMP_SIZES = [25, 50, 100]               # (c) comparison sizes
B_LIST = [1, 5, 10, 25, 50, 100, 200]   # only used if Q1 logs are missing
TIMING_REPEATS = 3                      # median fit time over this many repeats
print("SEED =", SEED)

In [ ]:
import os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.tree import DecisionTreeClassifier

for d in ["data", "split", "logs", "figures"]:
    os.makedirs(d, exist_ok=True)

faces = fetch_olivetti_faces(data_home="data/sklearn_data", shuffle=False)
X = faces.data.astype(np.float64); y = faces.target.astype(int)
assert X.shape == (400, 4096)

# ---- load the SAME saved split used in Q1 (or create it with the identical procedure) ----
for cand in ["split/split_indices.npz", "split_indices.npz"]:
    if os.path.exists(cand):
        s = np.load(cand); tr_idx, va_idx, te_idx = s["train"], s["val"], s["test"]
        print("Loaded split from", cand)
        if "seed" in s.files and int(s["seed"]) != SEED: print("WARNING: split seed differs from SEED")
        break
else:
    rng = np.random.default_rng(SEED); tr, va, te = [], [], []
    for k in range(K):
        p = rng.permutation(np.where(y == k)[0])
        tr += list(p[:6]); va += list(p[6:8]); te += list(p[8:])
    tr_idx, va_idx, te_idx = np.array(tr), np.array(va), np.array(te)
    np.savez("split/split_indices.npz", train=tr_idx, val=va_idx, test=te_idx, seed=SEED)
    print("Created split")
assert len(set(tr_idx) | set(va_idx) | set(te_idx)) == 400

def mask_images(A):
    B = A.copy().reshape(-1, 64, 64); B[:, 24:40, 24:40] = 0.0
    return B.reshape(len(A), -1)

Xtr, ytr = X[tr_idx], y[tr_idx]; Xva, yva = X[va_idx], y[va_idx]; Xte, yte = X[te_idx], y[te_idx]
Xte_m = mask_images(Xte)                # masked test set; all models are trained on clean images only
in_block = np.zeros((64, 64), bool); in_block[24:40, 24:40] = True; in_block = in_block.ravel()   # pixels hidden by the mask
print(Xtr.shape, Xva.shape, Xte.shape, Xte_m.shape)

In [ ]:
# ---- helpers from Q1 (bagging is my own implementation) ----
class Bagging:
    def __init__(self, n_estimators, tree_params, seed):
        self.B, self.tree_params, self.seed = n_estimators, tree_params, seed
    def fit(self, X, y):
        rng = np.random.default_rng(self.seed); n = len(X)
        self.trees, self.bags = [], []
        for _ in range(self.B):
            idx = rng.integers(0, n, size=n)
            t = DecisionTreeClassifier(**self.tree_params, random_state=int(rng.integers(2**31 - 1)))
            self.trees.append(t.fit(X[idx], y[idx])); self.bags.append(idx)
        return self
    def tree_preds(self, X):
        return np.stack([t.predict(X) for t in self.trees])
    def votes(self, X):
        return votes_from_preds(self.tree_preds(X))

def votes_from_preds(P, n_classes=K):
    V = np.zeros((P.shape[1], n_classes), int)
    for p in P: V[np.arange(len(p)), p] += 1
    return V

def disagreement_per_sample(P):
    # fraction of tree pairs that disagree, for each sample: (B^2 - sum_k c_k^2) / (B (B-1))
    B = P.shape[0]
    if B < 2: return np.full(P.shape[1], np.nan)
    V = votes_from_preds(P)
    return (B**2 - (V**2).sum(1)) / (B * (B - 1))

def load_or_compute_q1():
    try:
        TP = json.load(open("logs/frozen_tree_params.json")); FB = int(json.load(open("logs/final_B.json"))["final_B"])
        return TP, FB, "loaded from Q1 logs"
    except FileNotFoundError:
        best, order = None, 0                       # same rule as Q1: val acc desc, #leaves asc, grid order
        for d in [2, 4, 8, 16, None]:
            for m in [1, 2, 4]:
                c = DecisionTreeClassifier(max_depth=d, min_samples_leaf=m, random_state=SEED).fit(Xtr, ytr)
                key = (-c.score(Xva, yva), c.get_n_leaves(), order); order += 1
                if best is None or key < best[0]: best = (key, d, m)
        TP = dict(max_depth=best[1], min_samples_leaf=best[2])
        bg = Bagging(200, TP, SEED).fit(Xtr, ytr); Pv = bg.tree_preds(Xva)
        va = [(votes_from_preds(Pv[:B]).argmax(1) == yva).mean() for B in B_LIST]
        return TP, B_LIST[int(np.argmax(va))], "recomputed with Q1 rules"

TREE_PARAMS, FINAL_B, src = load_or_compute_q1()
print("Frozen single-tree params:", TREE_PARAMS, "| final bagging B:", FINAL_B, "|", src)

## SAMME implementation (from scratch)
* weights start at 1/n; each round trains a depth-limited `DecisionTreeClassifier` with `sample_weight=w`
* weighted error `eps = sum(w * wrong) / sum(w)`; `alpha = log((1-eps)/eps) + log(K-1)`; `w <- w * exp(alpha * wrong)`, renormalised
* **Weak learner not better than random multiclass guessing** (`eps >= 1 - 1/K`, i.e. `alpha <= 0`): the learner is **discarded and boosting stops**; the event is logged.
* **Perfect learner** (`eps = 0`): `alpha` would be infinite, so `eps` is clipped to 1e-10; the learner is kept and boosting stops (weights would no longer change).
* vote `S_k(x) = sum_t alpha_t 1[h_t(x)=k]`, confidence `q_k = S_k / sum_t alpha_t`.

In [ ]:
class SAMME:
    def __init__(self, n_estimators, max_depth, n_classes, seed, eps_floor=1e-10):
        self.T, self.depth, self.K, self.seed, self.eps_floor = n_estimators, max_depth, n_classes, seed, eps_floor
    def fit(self, X, y):
        n, K = len(y), self.K
        w = np.full(n, 1.0 / n)
        self.trees, self.alphas, self.log, self.events = [], [], [], []
        for t in range(self.T):
            tree = DecisionTreeClassifier(max_depth=self.depth, random_state=self.seed + t)
            tree.fit(X, y, sample_weight=w)
            pred = tree.predict(X); wrong = pred != y
            eps = float(np.sum(w * wrong) / np.sum(w))
            if eps >= 1.0 - 1.0 / K:            # not better than random K-class guessing
                self.events.append(f"round {t+1}: weighted error {eps:.4f} >= 1-1/K = {1-1/K:.4f}; learner discarded, boosting stopped")
                break
            eps_c = max(eps, self.eps_floor)
            alpha = float(np.log((1 - eps_c) / eps_c) + np.log(K - 1))
            w_new = w * np.exp(alpha * wrong); w_new = w_new / w_new.sum()
            self.log.append(dict(round=t + 1, w_before=w.copy(), pred=pred.copy(), wrong=wrong.copy(),
                                 eps=eps, alpha=alpha, w_after=w_new.copy()))
            self.trees.append(tree); self.alphas.append(alpha)
            if eps == 0.0:
                self.events.append(f"round {t+1}: perfect weak learner (eps=0); kept with clipped eps, boosting stopped")
                break
            w = w_new
        self.alphas = np.array(self.alphas)
        return self
    def learner_preds(self, X):                  # (n_learners, n_samples)
        return np.stack([t.predict(X) for t in self.trees])

def samme_scores(P, alphas, n, K=K):             # S_k(x) using the first n learners
    S = np.zeros((P.shape[1], K))
    for p, a in zip(P[:n], alphas[:n]): S[np.arange(len(p)), p] += a
    return S

def samme_q(P, alphas, n, K=K):                  # q_k = S_k / sum_t alpha_t
    return samme_scores(P, alphas, n, K) / np.sum(alphas[:n])

## Q2(a) Toy 3-class example: one complete SAMME round

In [ ]:
# 9 samples, 3 classes, 2 features. A depth-1 stump cannot separate them perfectly.
Xt = np.array([[1.0, 1.0], [1.5, 2.0], [3.5, 1.5],      # class 0
               [4.0, 4.0], [4.5, 3.5], [2.0, 3.5],      # class 1
               [1.0, 4.0], [1.5, 4.5], [4.0, 1.0]])     # class 2
yt = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])

toy = SAMME(n_estimators=1, max_depth=1, n_classes=3, seed=SEED).fit(Xt, yt)
r = toy.log[0]
stump = toy.trees[0].tree_
print(f"Stump split: feature x{stump.feature[0]+1} <= {stump.threshold[0]:.3f}")
toy_df = pd.DataFrame({"sample": range(9), "x1": Xt[:, 0], "x2": Xt[:, 1], "y": yt,
                       "w_initial": r["w_before"], "weak_pred": r["pred"], "wrong": r["wrong"].astype(int),
                       "w_updated_normalised": r["w_after"]})
display(toy_df.round(4))
print(f"weighted error eps = {r['eps']:.4f}   (random-guess limit 1-1/K = {1-1/3:.4f})")
print(f"alpha = log((1-eps)/eps) + log(K-1) = {r['alpha']:.4f}")

# independent hand-check of the update rule
w0 = np.full(9, 1 / 9); wrong = r["pred"] != yt
eps = (w0 * wrong).sum() / w0.sum(); alpha = np.log((1 - eps) / eps) + np.log(3 - 1)
w1 = w0 * np.exp(alpha * wrong); w1 /= w1.sum()
assert np.isclose(eps, r["eps"]) and np.isclose(alpha, r["alpha"]) and np.allclose(w1, r["w_after"])
print("hand-check passed; sum of updated weights =", r["w_after"].sum())
toy_df.to_csv("logs/q2a_toy_round.csv", index=False)
json.dump(dict(eps=r["eps"], alpha=r["alpha"], stump_feature=int(stump.feature[0]), stump_threshold=float(stump.threshold[0])),
          open("logs/q2a_toy_round.json", "w"))

## Q2(b) Boosting on Olivetti: depth and rounds chosen on clean validation accuracy only

In [ ]:
models, rows = {}, []
for d in DEPTHS:
    m = SAMME(T_MAX, d, K, SEED).fit(Xtr, ytr); models[d] = m
    Ptr_, Pva_ = (m.learner_preds(A) if len(m.trees) else None for A in (Xtr, Xva))
    print(f"depth {d}: {len(m.trees)} learners kept", m.events)
    for R in ROUNDS:
        n = min(R, len(m.trees))
        if n == 0: tr_acc = va_acc = float("nan")
        else:
            tr_acc = (samme_scores(Ptr_, m.alphas, n).argmax(1) == ytr).mean()
            va_acc = (samme_scores(Pva_, m.alphas, n).argmax(1) == yva).mean()
        rows.append(dict(depth=d, rounds=R, effective_rounds=n, train_acc=tr_acc, val_acc=va_acc))
grid2 = pd.DataFrame(rows); grid2.to_csv("logs/q2b_grid.csv", index=False)
display(grid2.round(4))

# Selection: highest validation accuracy; ties -> smaller depth -> fewer rounds
best = grid2.fillna(-1).sort_values(["val_acc", "depth", "rounds"], ascending=[False, True, True]).iloc[0]
SEL_D, SEL_R = int(best.depth), int(best.rounds)
boost_model = models[SEL_D]; N_SEL = min(SEL_R, len(boost_model.trees))
json.dump(dict(depth=SEL_D, rounds=SEL_R, effective_rounds=N_SEL, val_acc=float(best.val_acc)), open("logs/q2_frozen_boosting.json", "w"))
print(f"FROZEN boosting: depth={SEL_D}, rounds={SEL_R} (effective {N_SEL}), val acc={best.val_acc:.4f}")

In [ ]:
# Diagnostics over rounds for the selected model
L = boost_model.log[:N_SEL]
rr = np.arange(1, N_SEL + 1)
eps_t = np.array([r["eps"] for r in L]); alpha_t = np.array([r["alpha"] for r in L])
W = np.array([r["w_before"] for r in L])                 # weights the learner of each round was trained on
max_w = W.max(1); top10 = np.sort(W, axis=1)[:, -10:].sum(1)
diag = pd.DataFrame(dict(round=rr, weighted_train_error=eps_t, alpha=alpha_t, max_sample_weight=max_w, top10_weight=top10))
diag.to_csv("logs/q2b_selected_diagnostics.csv", index=False)

fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
for a, (col, ttl) in zip(ax, [("weighted_train_error", "weighted training error"), ("alpha", "learner weight alpha_t"),
                              ("max_sample_weight", "max sample weight"), ("top10_weight", "weight of 10 heaviest images")]):
    a.plot(diag["round"], diag[col]); a.set_title(ttl, fontsize=9); a.set_xlabel("round"); a.grid(alpha=.3)
ax[2].axhline(1 / len(ytr), ls=":", c="k"); ax[3].axhline(10 / len(ytr), ls=":", c="k")   # uniform references
plt.tight_layout(); plt.savefig("figures/q2b_diagnostics.pdf"); plt.show()

fig, a = plt.subplots(figsize=(4.5, 3.2))
for d in DEPTHS:
    g = grid2[grid2.depth == d]; a.plot(g.rounds, g.val_acc, "o-", label=f"depth {d}")
a.set_xlabel("rounds"); a.set_ylabel("validation accuracy"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("figures/q2b_val_grid.pdf"); plt.show()

## Q2(c) Boosting vs bagging at 25, 50, 100 learners

In [ ]:
def timed(fn, repeats=TIMING_REPEATS):
    ts, obj = [], None
    for _ in range(repeats):
        t0 = time.perf_counter(); o = fn(); ts.append(time.perf_counter() - t0)
        if obj is None: obj = o
    return obj, float(np.median(ts))

cmp_rows = []
for n_l in CMP_SIZES:
    bg, tb = timed(lambda: Bagging(n_l, TREE_PARAMS, SEED).fit(Xtr, ytr))
    accs = [(bg.votes(A).argmax(1) == yy).mean() for A, yy in [(Xtr, ytr), (Xva, yva), (Xte, yte)]]
    cmp_rows.append(dict(model=f"bagging (tree {TREE_PARAMS})", n_learners=n_l, learners_kept=n_l,
                         train_acc=accs[0], val_acc=accs[1], test_acc=accs[2], fit_time_s=tb))
    bs, tbs = timed(lambda: SAMME(n_l, SEL_D, K, SEED).fit(Xtr, ytr))
    k_ = len(bs.trees)
    if k_:
        accs = [(samme_scores(bs.learner_preds(A), bs.alphas, k_).argmax(1) == yy).mean() for A, yy in [(Xtr, ytr), (Xva, yva), (Xte, yte)]]
    else: accs = [float("nan")] * 3
    cmp_rows.append(dict(model=f"boosting SAMME (depth {SEL_D})", n_learners=n_l, learners_kept=k_,
                         train_acc=accs[0], val_acc=accs[1], test_acc=accs[2], fit_time_s=tbs))
cmp_df = pd.DataFrame(cmp_rows); cmp_df.to_csv("logs/q2c_bagging_vs_boosting.csv", index=False)
display(cmp_df.round(4))

In [ ]:
# One training face with a very large boosting weight, and its prediction history
Ptr_sel = boost_model.learner_preds(Xtr)[:N_SEL]
i_star = int(np.unravel_index(W.argmax(), W.shape)[1])          # face that reaches the largest weight in any round
r_peak = int(W[:, i_star].argmax()) + 1
y_i = int(ytr[i_star])
hist_pred = Ptr_sel[:, i_star]; correct = hist_pred == y_i
ratio = np.array([r["w_after"][i_star] / r["w_before"][i_star] for r in L])      # weight change factor each round
C = np.zeros((N_SEL, K)); C[np.arange(N_SEL), hist_pred] = alpha_t
q_hist = C.cumsum(0) / np.cumsum(alpha_t)[:, None]               # cumulative normalised vote after each round
q_true = q_hist[:, y_i]; q_other = np.delete(q_hist, y_i, axis=1).max(1)
wrong_labels = hist_pred[~correct]
top_wrong = int(np.bincount(wrong_labels, minlength=K).argmax()) if len(wrong_labels) else -1

print(f"train face {i_star} (dataset idx {tr_idx[i_star]}), true identity {y_i}")
print(f"uniform weight 1/n = {1/len(ytr):.4f}; peak weight {W[:, i_star].max():.4f} at round {r_peak} "
      f"({W[:, i_star].max() * len(ytr):.1f}x uniform); rank by final weight: {int((W[-1] > W[-1, i_star]).sum()) + 1}")
print(f"rounds correct: {correct.sum()}/{N_SEL}; most frequent wrong label: {top_wrong} ({(hist_pred == top_wrong).sum()} rounds)")
print(f"mean weight factor after a wrong round: {ratio[~correct].mean() if (~correct).any() else float('nan'):.3f}; after a correct round: {ratio[correct].mean() if correct.any() else float('nan'):.3f}")
print(f"final q(true)={q_true[-1]:.3f}, final q(best other)={q_other[-1]:.3f}")
print("first 15 rounds (pred, correct, weight before):")
display(pd.DataFrame(dict(round=rr[:15], pred=hist_pred[:15], correct=correct[:15], weight=W[:15, i_star], alpha=alpha_t[:15])).round(4))

fig, ax = plt.subplots(1, 4, figsize=(16, 3.3))
ax[0].plot(rr, W[:, i_star]); ax[0].scatter(rr[~correct], W[~correct, i_star], c="r", s=14, label="h_t wrong")
ax[0].scatter(rr[correct], W[correct, i_star], c="g", s=14, label="h_t right"); ax[0].axhline(1 / len(ytr), ls=":", c="k")
ax[0].set_title("weight of the face by round", fontsize=9); ax[0].set_xlabel("round"); ax[0].legend(fontsize=7)
ax[1].plot(rr, q_true, label="true class"); ax[1].plot(rr, q_other, label="best other class"); ax[1].set_title("cumulative vote share", fontsize=9)
ax[1].set_xlabel("round"); ax[1].legend(fontsize=7)
ax[2].imshow(Xtr[i_star].reshape(64, 64), cmap="gray"); ax[2].set_title(f"heavy face (true {y_i})", fontsize=9); ax[2].axis("off")
if top_wrong >= 0:
    ax[3].imshow(Xtr[np.where(ytr == top_wrong)[0][0]].reshape(64, 64), cmap="gray"); ax[3].set_title(f"often predicted: {top_wrong}", fontsize=9)
ax[3].axis("off")
plt.tight_layout(); plt.savefig("figures/q2c_heavy_face.pdf"); plt.show()
json.dump(dict(train_pos=i_star, dataset_idx=int(tr_idx[i_star]), true=y_i, peak_weight=float(W[:, i_star].max()), peak_round=r_peak,
               rounds_correct=int(correct.sum()), rounds=N_SEL, most_frequent_wrong=top_wrong,
               mean_factor_wrong=float(ratio[~correct].mean()) if (~correct).any() else None,
               mean_factor_correct=float(ratio[correct].mean()) if correct.any() else None,
               pred_history=hist_pred.tolist()), open("logs/q2c_heavy_face.json", "w"), indent=1)

## Q2(d) Frozen single tree vs final bagged model vs frozen boosting model (clean and masked test)

In [ ]:
tree = DecisionTreeClassifier(**TREE_PARAMS, random_state=SEED).fit(Xtr, ytr)      # frozen single tree (same as Q1)
bag = Bagging(FINAL_B, TREE_PARAMS, SEED).fit(Xtr, ytr)                            # final bagged model (first FINAL_B trees of Q1)
assert (tree.classes_ == np.arange(K)).all()

Pbag = {"clean": bag.tree_preds(Xte), "masked": bag.tree_preds(Xte_m)}
Pbst = {"clean": boost_model.learner_preds(Xte)[:N_SEL], "masked": boost_model.learner_preds(Xte_m)[:N_SEL]}
dist = {}                                   # per model/test set: (n_test, K) distribution; winning value = confidence
for nm, A in [("clean", Xte), ("masked", Xte_m)]:
    dist[("single tree", nm)] = tree.predict_proba(A)
    dist[("bagging", nm)] = votes_from_preds(Pbag[nm]) / FINAL_B
    dist[("boosting", nm)] = samme_q(Pbst[nm], boost_model.alphas, N_SEL)
pred = {k: v.argmax(1) for k, v in dist.items()}
rows = []
for (mdl, nm), D in dist.items():
    p = pred[(mdl, nm)]; c = D.max(1)
    rows.append(dict(model=mdl, test_set=nm, accuracy=(p == yte).mean(), mean_winning_vote_or_prob=c.mean(),
                     conf_when_correct=c[p == yte].mean() if (p == yte).any() else np.nan,
                     conf_when_wrong=c[p != yte].mean() if (p != yte).any() else np.nan))
q2d = pd.DataFrame(rows).sort_values(["test_set", "model"]); q2d.to_csv("logs/q2d_clean_vs_masked.csv", index=False)
print(f"single tree {TREE_PARAMS}; bagging B={FINAL_B}; boosting depth={SEL_D}, {N_SEL} learners")
display(q2d.round(4))

# How much does each model look at the hidden block? (share of split nodes on block pixels; uniform = 6.25%)
def block_share(tree_):
    f = tree_.tree_.feature; f = f[f >= 0]
    return in_block[f].mean() if len(f) else 0.0
share = dict(single_tree=block_share(tree), bagging=float(np.mean([block_share(t) for t in bag.trees])),
             boosting_alpha_weighted=float(np.average([block_share(t) for t in boost_model.trees[:N_SEL]], weights=alpha_t)))
print("share of splits on masked-block pixels:", {k: round(v, 4) for k, v in share.items()})

In [ ]:
# One face where bagging and boosting respond differently to the missing region
yy = yte
bagC, bagM, bstC, bstM = (pred[("bagging", "clean")], pred[("bagging", "masked")], pred[("boosting", "clean")], pred[("boosting", "masked")])
drop_bag = dist[("bagging", "clean")][np.arange(len(yy)), yy] - dist[("bagging", "masked")][np.arange(len(yy)), yy]
drop_bst = dist[("boosting", "clean")][np.arange(len(yy)), yy] - dist[("boosting", "masked")][np.arange(len(yy)), yy]
gap = drop_bst - drop_bag                         # >0: boosting loses more true-class support than bagging
cats = [("bagging stays correct, boosting becomes wrong", (bagC == yy) & (bagM == yy) & (bstC == yy) & (bstM != yy)),
        ("boosting stays correct, bagging becomes wrong", (bstC == yy) & (bstM == yy) & (bagC == yy) & (bagM != yy)),
        ("bagging prediction changes, boosting does not", (bagC != bagM) & (bstC == bstM)),
        ("boosting prediction changes, bagging does not", (bstC != bstM) & (bagC == bagM))]
chosen = None
for name, mask in cats:
    if mask.any():
        idxs = np.where(mask)[0]; chosen = (name, int(idxs[np.argmax(np.abs(gap[idxs]))])); break
if chosen is None:
    chosen = ("no category matched; face with the largest difference in true-class support loss", int(np.argmax(np.abs(gap))))
cat_name, f = chosen
print("Category:", cat_name, "| test position", f, "| dataset idx", int(te_idx[f]), "| true", int(yy[f]))

def top3(v, scale=1.0): o = np.argsort(-v)[:3]; return {int(k): round(float(v[k] * scale), 3) for k in o}
print("tree     clean pred", int(pred[("single tree", "clean")][f]), "| masked pred", int(pred[("single tree", "masked")][f]))
print("bagging  votes clean :", top3(dist[("bagging", "clean")][f], FINAL_B), "| masked:", top3(dist[("bagging", "masked")][f], FINAL_B))
print("boosting q clean     :", top3(dist[("boosting", "clean")][f]), "| masked:", top3(dist[("boosting", "masked")][f]))

def face_disagree(P, j): return float(disagreement_per_sample(P[:, [j]])[0])
chg_bag = (Pbag["clean"][:, f] != Pbag["masked"][:, f]); chg_bst = (Pbst["clean"][:, f] != Pbst["masked"][:, f])
def uses_block(tree_, x):
    nodes = tree_.decision_path(x.reshape(1, -1)).indices; ft = tree_.tree_.feature[nodes]; ft = ft[ft >= 0]
    return bool(in_block[ft].any())
ub_bag = np.array([uses_block(t, Xte[f]) for t in bag.trees]); ub_bst = np.array([uses_block(t, Xte[f]) for t in boost_model.trees[:N_SEL]])
info = dict(category=cat_name, test_pos=f, dataset_idx=int(te_idx[f]), true=int(yy[f]),
    bagging_pairwise_disagreement_clean=face_disagree(Pbag["clean"], f), bagging_pairwise_disagreement_masked=face_disagree(Pbag["masked"], f),
    boosting_pairwise_disagreement_clean=face_disagree(Pbst["clean"], f), boosting_pairwise_disagreement_masked=face_disagree(Pbst["masked"], f),
    bagging_frac_trees_changed=float(chg_bag.mean()), boosting_alpha_weighted_frac_learners_changed=float(np.average(chg_bst, weights=alpha_t)),
    bagging_frac_trees_whose_path_uses_block=float(ub_bag.mean()), boosting_alpha_weighted_frac_paths_use_block=float(np.average(ub_bst, weights=alpha_t)),
    bagging_true_class_support_clean_masked=[float(dist[("bagging", "clean")][f, yy[f]]), float(dist[("bagging", "masked")][f, yy[f]])],
    boosting_true_class_support_clean_masked=[float(dist[("boosting", "clean")][f, yy[f]]), float(dist[("boosting", "masked")][f, yy[f]])],
    block_share_of_splits={k: float(v) for k, v in share.items()})
for k, v in info.items(): print(f"  {k}: {v}")
json.dump(info, open("logs/q2d_case.json", "w"), indent=1)

def ex(c): return Xtr[np.where(ytr == c)[0][0]].reshape(64, 64)
fig, ax = plt.subplots(1, 5, figsize=(12, 2.8))
for a, (im, t) in zip(ax, [(Xte[f].reshape(64, 64), f"clean, true {yy[f]}"), (Xte_m[f].reshape(64, 64), "masked"),
                           (ex(yy[f]), "true class example"), (ex(bagM[f]), f"bagging masked pred {bagM[f]}"), (ex(bstM[f]), f"boosting masked pred {bstM[f]}")]):
    a.imshow(im, cmap="gray"); a.set_title(t, fontsize=8); a.axis("off")
plt.tight_layout(); plt.savefig("figures/q2d_case.pdf"); plt.show()

In [ ]:
final = dict(roll_no=ROLL_NO, seed=SEED, frozen_tree=TREE_PARAMS, final_B=FINAL_B,
             frozen_boosting=dict(depth=SEL_D, rounds=SEL_R, effective_rounds=N_SEL),
             boosting_events={d: m.events for d, m in models.items()},
             q2c=cmp_df.to_dict(orient="records"), q2d=q2d.to_dict(orient="records"))
json.dump(final, open("logs/q2_final_log.json", "w"), indent=2, default=str)
print("saved logs/q2_final_log.json")